# Stage 3 — DINOv2 + fingertip-region pooled (NO landmark features)

Stage 2 mean-pooled all 256 patch tokens and hit CER 0.86 — much worse than Stage 1 landmarks (0.69). Diagnosis: mean-pool destroys spatial focus.

Stage 3 swaps mean-pool for a **2×2 patch window centered on the MediaPipe fingertip**. Same DINOv2 backbone, same Conformer head, same split, only the spatial reduction changes.

**Pipeline** per clip:
  1. Detect 21 landmarks per frame (MediaPipe)
  2. Union bbox → crop+resize to 224×224
  3. DINOv2 forward → [T, 256, 384] patch tokens
  4. Transform fingertip to cropped coords → 2×2 patch block (4 tokens)
  5. Mean-pool those 4 tokens → [384]
  6. Resample to T_native=32

**Pass/fail (per plan)**:
- ≤ Stage 2 (≤ 0.86) → spatial conditioning helped, continue
- ≤ Stage 1 (≤ 0.69) → matches/beats trajectory → Stage 4 fusion is the main event
- > Stage 2 → frozen DINOv2 fundamentally wrong; skip to Stage 4

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' --quiet

import sys, os
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

## Cell 2 — Config

In [ ]:
import os, logging, random, json
import numpy as np
import torch

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs',        exist_ok=True)
os.makedirs('/kaggle/working/manifests',   exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s \u2014 %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/kaggle/working/logs/stage3.log'),
    ]
)

from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig,
)

STAGE3_CACHE   = '/kaggle/working/dinov2_fingertip_features_t32.pt'
SPLIT_MANIFEST = '/kaggle/working/manifests/subject_split.json'

# Hyperparams: identical to Stages 1-2.  Only the encoder feature changes.
T_NATIVE        = 32
UPSAMPLE        = 2
D_MODEL         = 256
N_LAYERS        = 4
N_HEADS         = 4
CONV_KERNEL     = 15
DROPOUT         = 0.2
BATCH_SIZE      = 32
LR_PEAK         = 5e-4
WEIGHT_DECAY    = 5e-2
GRAD_CLIP       = 1.0
NUM_EPOCHS      = 80
WARMUP_PCT      = 0.05
SEED            = 42

USE_AUGMENT     = True
DINOV2_MODEL    = 'facebook/dinov2-small'
DINOV2_IMG_SIZE = 224
WINDOW          = 2          # 2×2 = 4 patches around fingertip
PADDING_RATIO   = 0.3

cfg = Config(
    data=DataConfig(
        hf_repo_id='yewon816/WiTA',
        lang='english',
        max_zips=None,
        max_frames=64,
        train_split=0.90,
        seed=SEED,
    ),
    encoder=EncoderConfig(arch='siglip'),    # placeholder, unused
    train=TrainConfig(
        num_epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR_PEAK,
        weight_decay=WEIGHT_DECAY,
        grad_clip=GRAD_CLIP,
        num_workers=2,
        warmup_pct=WARMUP_PCT,
        seed=SEED,
        checkpoint_dir='/kaggle/working/checkpoints',
    ),
).build()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

print(f'Device       : {cfg.device}')
print(f'Vocab        : {cfg.vocab.ctc_vocab_size}')
print(f'DINOv2       : {DINOV2_MODEL}  img={DINOV2_IMG_SIZE}  pool=fingertip_{WINDOW}x{WINDOW}')
print(f'T_native     : {T_NATIVE}   T_out = {T_NATIVE * UPSAMPLE}')
print(f'Regulariz.   : dropout={DROPOUT}  weight_decay={WEIGHT_DECAY}  epochs={NUM_EPOCHS}')
print(f'Augmentation : {USE_AUGMENT}')

## Cell 3 — Stream + index

In [ ]:
from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
samples = stream_and_index_with_subjects(cfg)
print(f'Total: {len(samples)} clips across {len({s for _,_,s in samples})} subjects')

## Cell 4 — Subject-disjoint split (same seed → same split as Stages 1, 2)

In [ ]:
from wita_v2.datasets.subject_splits import build_subject_split, save_split_manifest, load_split_manifest

if os.path.exists(SPLIT_MANIFEST):
    manifest = load_split_manifest(SPLIT_MANIFEST)
    print(f'Loaded manifest from {SPLIT_MANIFEST}')
else:
    _, _, manifest = build_subject_split(samples, train_ratio=0.90, seed=SEED)
    save_split_manifest(manifest, SPLIT_MANIFEST)

print(f'Train subjects: {manifest["n_subjects_train"]}   clips: {manifest["n_train_clips"]}')
print(f'Val   subjects: {manifest["n_subjects_val"]}     clips: {manifest["n_val_clips"]}')
print(f'Val subjects: {manifest["val_subjects"]}')

## Cell 5 — Extract fingertip-pooled DINOv2 features (ONCE, ~30-40 min on T4)

Slower than Stage 2 because we run MediaPipe per frame in addition to DINOv2. Detect rate and fallback-frame stats printed at the end — high fallback count means MediaPipe is struggling on some clips.

In [ ]:
from wita_v2.models.encoders.dinov2_encoder    import DINOv2Encoder
from wita_v2.datasets.dinov2_fingertip_cache import extract_dinov2_fingertip_features

if os.path.exists(STAGE3_CACHE):
    print(f'Cache exists: {STAGE3_CACHE}  ({os.path.getsize(STAGE3_CACHE)/1e6:.1f} MB)')
else:
    encoder = DINOv2Encoder(
        model_name = DINOV2_MODEL,
        image_size = DINOV2_IMG_SIZE,
        pool       = 'mean_patch',     # only used by .forward(); forward_patches bypasses
    )
    extract_dinov2_fingertip_features(
        samples       = samples,
        encoder       = encoder,
        out_path      = STAGE3_CACHE,
        T_native      = T_NATIVE,
        padding_ratio = PADDING_RATIO,
        window        = WINDOW,
        seg_chunk     = 16,
        device        = cfg.device,
        dtype         = torch.float16,
    )
    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Cell 6 — Dataloaders + augmentation

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from wita_v2.datasets.vocab import make_converter
from wita_v2.datasets.skeleton_augment import LandmarkAugment

cache = torch.load(STAGE3_CACHE, map_location='cpu', weights_only=False)
converter = make_converter(cfg.data.lang)

subject_per_idx = cache['subjects']
unique_subjects = sorted(set(subject_per_idx))
print(f'Cache subjects ({len(unique_subjects)})')
print(f'Detect rate (frames): {cache.get("frame_detect_rate", 0)*100:.1f}%')
print(f'Fallback frames     : {cache.get("fallback_frames_total", 0)}')

train_set, val_set = set(manifest['train_subjects']), set(manifest['val_subjects'])
train_idx_cache = [i for i, s in enumerate(subject_per_idx) if s in train_set]
val_idx_cache   = [i for i, s in enumerate(subject_per_idx) if s in val_set]
print(f'Cache: {len(cache["feats"])} clips, out_dim={cache["out_dim"]}')
print(f'Train: {len(train_idx_cache)}   Val: {len(val_idx_cache)}')

if len(val_idx_cache) == 0:
    raise RuntimeError('VAL SET IS EMPTY.')

class FeatureDataset(Dataset):
    def __init__(self, cache, indices, converter, transform=None):
        self.cache = cache; self.indices = indices
        self.converter = converter; self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        feats = self.cache['feats'][idx].float()
        if self.transform is not None:
            feats = self.transform(feats)
        label_str = self.cache['labels'][idx]
        enc, _ = self.converter.encode(label_str)
        return feats, enc

def collate(batch, pad_idx):
    feats, labels = zip(*batch)
    feats_pad  = pad_sequence(feats, batch_first=True, padding_value=0.0)
    labels_pad = pad_sequence(labels, batch_first=True, padding_value=pad_idx)
    input_lens = torch.LongTensor([f.shape[0] for f in feats])
    label_lens = torch.LongTensor([l.shape[0] for l in labels])
    return feats_pad, labels_pad, input_lens, label_lens

pad_idx = cfg.vocab.pad_idx
collate_fn = lambda b: collate(b, pad_idx=pad_idx)

# Same temporal-only augmentation as Stage 2.
train_aug = LandmarkAugment(p_spatial_jitter=0.0) if USE_AUGMENT else None
val_aug   = None
print(f'Train augment: {train_aug}')

train_ds = FeatureDataset(cache, train_idx_cache, converter, transform=train_aug)
val_ds   = FeatureDataset(cache, val_idx_cache,   converter, transform=val_aug)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)
print(f'Train batches: {len(train_loader)}   Val batches: {len(val_loader)}')

feats, labels, in_lens, lab_lens = next(iter(train_loader))
print(f'feats: {feats.shape}  in_lens[:4]={in_lens[:4].tolist()}  lab_lens[:4]={lab_lens[:4].tolist()}')

## Cell 7 — Build ConformerCTC

In [ ]:
from wita_v2.models.conformer_ctc import ConformerCTC
from wita_v2.training.diagnostics import assert_ctc_feasible

model = ConformerCTC(
    input_dim   = cache['out_dim'],      # 384
    vocab_size  = cfg.vocab.ctc_vocab_size,
    d_model     = D_MODEL,
    n_layers    = N_LAYERS,
    n_heads     = N_HEADS,
    conv_kernel = CONV_KERNEL,
    dropout     = DROPOUT,
    upsample    = UPSAMPLE,
).to(cfg.device)
print(f'Trainable params: {model.num_params:,}')

with torch.no_grad():
    lp, enc_lens = model(feats.to(cfg.device), in_lens.to(cfg.device))
print(f'log_probs: {lp.shape}   enc_lens[:4]={enc_lens[:4].tolist()}')
stats = assert_ctc_feasible(enc_lens.cpu(), lab_lens, raise_on_fail=False)
print(f'CTC feasibility: feasible={stats["feasible"]}/{stats["total"]}  infeasible={stats["infeasible"]}')

import torch.nn as nn
ctc = nn.CTCLoss(blank=cfg.vocab.blank_idx, zero_infinity=True, reduction='mean')
with torch.no_grad():
    loss = ctc(lp.transpose(0,1).float(), labels.to(cfg.device), enc_lens, lab_lens.to(cfg.device))
print(f'Initial CTC loss: {loss.item():.4f}')

## Cell 8 — Train

In [ ]:
import time
from wita_v2.training.diagnostics import (
    full_diagnostic_snapshot, format_snapshot_line, assert_ctc_feasible,
)

device = cfg.device
assert len(val_loader) > 0, 'val_loader is empty'

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PEAK,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
total_steps = NUM_EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_PEAK, total_steps=total_steps,
    pct_start=WARMUP_PCT, anneal_strategy='cos',
)

def decode_argmax(log_probs, enc_lens, blank):
    out = []
    argmax = log_probs.argmax(dim=-1)
    for b in range(argmax.shape[0]):
        seq = argmax[b, :int(enc_lens[b].item())].tolist()
        merged = []; prev = None
        for t in seq:
            if t != prev and t != blank: merged.append(t)
            prev = t
        out.append(merged)
    return out

best_cer = float('inf')
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    sum_loss = 0.0; n_batches = 0; t0 = time.time()
    for feats, labels, in_lens, lab_lens in train_loader:
        feats = feats.to(device); labels = labels.to(device)
        in_lens = in_lens.to(device); lab_lens = lab_lens.to(device)
        lp, enc_lens = model(feats, in_lens)
        assert_ctc_feasible(enc_lens.cpu(), lab_lens.cpu(), raise_on_fail=True)
        loss = ctc(lp.transpose(0,1).float(), labels, enc_lens, lab_lens)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step(); scheduler.step()
        sum_loss += loss.item(); n_batches += 1
    train_loss = sum_loss / max(n_batches, 1)

    model.eval()
    pairs = []; sum_val_loss = 0.0; n_val_batches = 0
    last_lp = last_lens = None
    with torch.no_grad():
        for feats, labels, in_lens, lab_lens in val_loader:
            feats = feats.to(device); labels = labels.to(device)
            in_lens = in_lens.to(device); lab_lens = lab_lens.to(device)
            lp, enc_lens = model(feats, in_lens)
            v_loss = ctc(lp.transpose(0,1).float(), labels, enc_lens, lab_lens)
            sum_val_loss += v_loss.item(); n_val_batches += 1
            for b, pred in enumerate(decode_argmax(lp, enc_lens, cfg.vocab.blank_idx)):
                gt = converter.decode(labels[b, :lab_lens[b].item()].int().cpu(),
                                       torch.IntTensor([lab_lens[b].item()]))
                pred_str = ''.join(cfg.vocab.chars[t-1] if 1<=t<=len(cfg.vocab.chars) else '?' for t in pred)
                pairs.append((gt, pred_str))
            last_lp = lp; last_lens = enc_lens
    val_loss = sum_val_loss / max(n_val_batches, 1)

    snap = full_diagnostic_snapshot(
        pairs=pairs, log_probs=last_lp, lengths=last_lens,
        chars=cfg.vocab.chars, train_loss=train_loss, val_loss=val_loss,
        blank=cfg.vocab.blank_idx,
    )
    history.append({'epoch': epoch+1, **{k: v for k, v in snap.items() if not isinstance(v, (list, dict))}})
    dt = time.time() - t0
    print(f'[Ep {epoch+1:3d}/{NUM_EPOCHS}] train={train_loss:.4f}  val_loss={val_loss:.4f}  '
          f'{format_snapshot_line(snap)}  {dt:.0f}s', flush=True)

    if snap['val_cer_overall'] < best_cer:
        best_cer = snap['val_cer_overall']
        torch.save({'model_state_dict': model.state_dict(),
                    'epoch': epoch, 'val_cer': best_cer, 'snapshot': snap},
                   '/kaggle/working/checkpoints/stage3_best.pt')
        print(f'  \u2605 new best CER={best_cer:.4f}')

print(f'\nStage 3 done. Best val CER: {best_cer:.4f}')
with open('/kaggle/working/logs/stage3_history.json', 'w') as f:
    json.dump(history, f, indent=2)

## Cell 9 — Verdict vs Stages 1 & 2

In [ ]:
ckpt = torch.load('/kaggle/working/checkpoints/stage3_best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
snap = ckpt['snapshot']

STAGE1_BEST_CER = 0.6870
STAGE2_BEST_CER = 0.8601
cer3 = snap['val_cer_overall']

print(f'\n=== STAGE 3 RESULT ===')
print(f'Best val CER: {cer3:.4f}')
print(f'  vs Stage 1 (landmarks):  {cer3 - STAGE1_BEST_CER:+.4f}')
print(f'  vs Stage 2 (mean-pool):  {cer3 - STAGE2_BEST_CER:+.4f}')
print()
print('Length-bucketed CER:')
for bucket, m in snap['length_buckets'].items():
    print(f'  L={bucket:5s}  n={m["n_clips"]:4d}  CER={m["cer"]:.4f}  '
          f'pred_len={m["mean_pred_len"]:.1f}  target_len={m["mean_target_len"]:.1f}')
print()
print(f'Edit decomp: {snap["edit_decomposition"]}')
print(f'NLL gap:     {snap.get("nll_gap", float("nan")):+.4f}')
print(f'KL:          {snap["kl_pred_vs_label_nats"]:.3f}')
print()
print('Pass/fail per plan:')
if cer3 <= STAGE1_BEST_CER:
    print(f'  \u2705 BEATS Stage 1 (\u2264 {STAGE1_BEST_CER:.4f}) \u2014 spatial focus on fingertip works. '
          'Proceed to Stage 4 (fusion).')
elif cer3 <= STAGE2_BEST_CER:
    print(f'  \u26a1 BETTER than Stage 2 ({STAGE2_BEST_CER:.4f}) but worse than Stage 1 '
          f'({STAGE1_BEST_CER:.4f}). Fingertip-pool helps but trajectory still dominates.')
else:
    print(f'  \u274c HURTS even more than Stage 2. Investigate: MediaPipe detect rate? '
          'Fingertip-vs-trajectory mismatch? Skip to Stage 4.')